# 02 — EDA & Featurization

Loads `../data/metals_dos.csv` (produced by `01_data_acquisition.ipynb`), explores the DOS(E_F)
distribution across elemental families and crystal systems, then featurizes each composition with
MAGPIE descriptors (via `matminer`) plus a one-hot crystal-system indicator. Output:
`../data/features.csv`, consumed by `03_modeling.ipynb`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pymatgen.core import Composition
from matminer.featurizers.composition import ElementProperty

print("Imports OK")


In [ ]:
df = pd.read_csv("../data/metals_dos.csv")
print(f"Loaded {len(df)} materials from ../data/metals_dos.csv")
df.head()


---
## Exploratory data analysis

A quick look at how many materials fall into each elemental family and crystal system, and the
shape of the DOS(E_F) distribution itself, before featurizing.

In [ ]:
print("Materials per elemental family:")
print(df["family"].value_counts())
print("\nMaterials per crystal system:")
print(df["crystal_system"].value_counts())
print("\nDOS(E_F) summary statistics:")
print(df["dos_ef"].describe())


In [ ]:
# DOS(E_F) is strictly positive and heavily right-skewed (a handful of large-DOS outliers,
# e.g. flat-band rare-earth compounds, alongside a bulk of modest values), so a log-scaled
# x-axis is what actually makes the distribution shape legible.
plt.figure(figsize=(8, 5))
sns.histplot(df["dos_ef"], bins=40, log_scale=True)
plt.xlabel("DOS(E_F)  (states / eV / formula unit, log scale)")
plt.title("Distribution of DOS(E_F) across sampled metals")
plt.tight_layout()
plt.show()


---
## Part 7 — Featurization: MAGPIE compositional descriptors + crystal system

We reconstruct each material's `Composition` from its `formula` string (MAGPIE features are purely
compositional, so this is equivalent to featurizing from the original `Structure` object) and run
matminer's MAGPIE `ElementProperty` featurizer (the same composition-based feature set used in
Ward et al. 2016), then add the crystal system as a one-hot indicator so the model can also use
coarse structural information without needing full structural descriptors. `family` and `period`
ride along unchanged for use as `GroupKFold` groups downstream.

In [ ]:
df["composition"] = df["formula"].apply(Composition)

ep_feat = ElementProperty.from_preset("magpie")
magpie_cols = ep_feat.feature_labels()

df_feat = ep_feat.featurize_dataframe(df, col_id="composition", ignore_errors=True)

xtal_dummies = pd.get_dummies(df_feat["crystal_system"], prefix="xtal")

X_full = pd.concat([df_feat[magpie_cols], xtal_dummies], axis=1)
y_full = df_feat["dos_ef"]

# Drop rows with missing features or missing target (failed featurization, if any)
mask = X_full.notna().all(axis=1) & y_full.notna()
X = X_full[mask].reset_index(drop=True)
y = y_full[mask].reset_index(drop=True)
meta = df_feat.loc[mask, ["material_id", "formula", "crystal_system", "family", "period"]].reset_index(drop=True)

print(f"Final modeling set: {len(X)} materials, {X.shape[1]} features "
      f"({len(magpie_cols)} MAGPIE + {len(xtal_dummies.columns)} crystal-system indicators)")


---
## Save output

Combine features, target, and metadata into a single tidy CSV for the modeling notebook.

In [ ]:
import os

df_features = pd.concat([meta, X, y.rename("dos_ef")], axis=1)

os.makedirs("../data", exist_ok=True)
df_features.to_csv("../data/features.csv", index=False)
print(f"Saved {len(df_features)} rows, {X.shape[1]} feature columns to ../data/features.csv")
df_features.head()
